# Neural Network Model
Standalone NN notebook — trains a 1D-CNN + MLP on raw sequences + engineered features.

**Outputs saved:**
- `nn_oof_preds.npy` — out-of-fold predictions (for stacking in your main notebook)
- `nn_test_preds.npy` — test predictions
- `nn_submission.csv` — standalone submission (optional)

To use in your main notebook, load these files and add `'nn'` to your stacking ensemble.

In [1]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import GroupKFold
from sklearn.metrics import mean_absolute_error
from sklearn.preprocessing import LabelEncoder, StandardScaler
from itertools import combinations
import warnings
warnings.filterwarnings('ignore')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

Using device: cpu


## 1. Load Data

In [2]:
train_raw = pd.read_csv('train_competition_2026.csv')
test_raw  = pd.read_csv('test_no_outcome.csv')

train_raw['time'] = pd.to_datetime(train_raw['time'])
test_raw['time']  = pd.to_datetime(test_raw['time'])

print(f'Train raw: {train_raw.shape}')
print(f'Test raw:  {test_raw.shape}')

Train raw: (432600, 18)
Test raw:  (103500, 16)


## 2. Feature Engineering (same as main notebook)

In [3]:
def engineer_features(df):
    t_cols   = [f't_{i}' for i in range(5)]
    num_cols = ['num_0', 'num_1', 'num_2']
    cat_cols = [f'cat_{i}' for i in range(5)]

    df = df.sort_values(['obs', 'time']).copy()

    agg_dict = {}
    for c in num_cols + cat_cols:
        agg_dict[c] = 'first'
    agg_dict['sub_id'] = 'first'
    for c in t_cols:
        agg_dict[c] = ['mean', 'std', 'min', 'max', 'first', 'last', 'median']

    grouped = df.groupby('obs').agg(agg_dict)
    grouped.columns = ['_'.join(col).strip('_') for col in grouped.columns]
    grouped = grouped.reset_index()

    for c in t_cols:
        grouped[f'{c}_slope'] = grouped[f'{c}_last'] - grouped[f'{c}_first']
        grouped[f'{c}_range'] = grouped[f'{c}_max']  - grouped[f'{c}_min']
        grouped[f'{c}_cv']    = grouped[f'{c}_std']  / (grouped[f'{c}_mean'].abs() + 1e-8)

    quantile_feats = df.groupby('obs')[t_cols].quantile([0.1, 0.25, 0.75, 0.9])
    quantile_feats = quantile_feats.unstack(level=-1)
    quantile_feats.columns = [f'{c}_q{int(q*100)}' for c, q in quantile_feats.columns]
    grouped = grouped.merge(quantile_feats.reset_index(), on='obs')

    skew_feats = df.groupby('obs')[t_cols].skew()
    skew_feats.columns = [f'{c}_skew' for c in t_cols]
    grouped = grouped.merge(skew_feats.reset_index(), on='obs')

    kurt_feats = df.groupby('obs')[t_cols].apply(lambda x: x.kurtosis())
    kurt_feats.columns = [f'{c}_kurt' for c in t_cols]
    grouped = grouped.merge(kurt_feats.reset_index(), on='obs')

    grouped['t0_minus_t1'] = grouped['t_0_mean'] - grouped['t_1_mean']
    grouped['t2_minus_t3'] = grouped['t_2_mean'] - grouped['t_3_mean']
    grouped['t_mean_all']  = grouped[[f't_{i}_mean' for i in range(5)]].mean(axis=1)
    grouped['num0_times_num1'] = grouped['num_0_first'] * grouped['num_1_first']
    grouped['num0_times_num2'] = grouped['num_0_first'] * grouped['num_2_first']

    time_feats = df.groupby('obs')['time'].first()
    grouped['hour']       = pd.to_datetime(time_feats.values).hour
    grouped['dayofweek']  = pd.to_datetime(time_feats.values).dayofweek
    grouped['is_weekend'] = (grouped['dayofweek'] >= 5).astype(int)

    df['_tsec'] = (df['time'] - df.groupby('obs')['time'].transform('first')).dt.total_seconds()
    meta = df.groupby('obs').agg(n_points=('time','size'), duration_seconds=('_tsec','max')).reset_index()
    grouped = grouped.merge(meta, on='obs', how='left')

    def dyn_stats(block):
        out = {}
        t = block['_tsec'].values.astype(float)
        n = len(t)
        if n < 2:
            for c in t_cols:
                x = block[c].values.astype(float)
                out[f'{c}_mean_abs_diff']  = 0.0
                out[f'{c}_last_minus_mean'] = float(x[-1] - np.mean(x)) if n == 1 else 0.0
                out[f'{c}_slope_lr']        = 0.0
            return pd.Series(out)
        tc    = t - t.mean()
        denom = float(np.sum(tc**2)) + 1e-12
        for c in t_cols:
            x  = block[c].values.astype(float)
            xc = x - x.mean()
            out[f'{c}_mean_abs_diff']  = float(np.mean(np.abs(np.diff(x))))
            out[f'{c}_last_minus_mean'] = float(x[-1] - np.mean(x))
            out[f'{c}_slope_lr']        = float(np.sum(tc * xc) / denom)
        return pd.Series(out)

    dyn = df.groupby('obs', sort=False).apply(dyn_stats).reset_index()
    grouped = grouped.merge(dyn, on='obs', how='left')
    df.drop(columns=['_tsec'], inplace=True)

    sub_counts = df.groupby('sub_id')['obs'].nunique().reset_index()
    sub_counts.columns = ['sub_id', 'sub_obs_count']
    grouped = grouped.merge(sub_counts, left_on='sub_id_first', right_on='sub_id', how='left')
    grouped = grouped.drop(columns=['sub_id'])

    # Cross-signal correlations
    def cross_correlations(block):
        out  = {}
        vals = {c: block[c].values.astype(float) for c in t_cols}
        for c1, c2 in combinations(t_cols, 2):
            v1, v2 = vals[c1], vals[c2]
            denom  = (np.std(v1) * np.std(v2)) + 1e-12
            out[f'{c1}_{c2}_corr'] = float(np.corrcoef(v1, v2)[0, 1]) if denom > 1e-10 else 0.0
        return pd.Series(out)

    corr_feats = df.groupby('obs', sort=False).apply(cross_correlations).reset_index()
    grouped = grouped.merge(corr_feats, on='obs', how='left')

    # FFT features
    def fft_feats(block):
        out = {}
        for c in t_cols:
            x   = block[c].values.astype(float)
            fft = np.abs(np.fft.rfft(x - x.mean()))
            out[f'{c}_fft_energy']    = float(np.sum(fft**2))
            out[f'{c}_fft_peak_freq'] = float(np.argmax(fft[1:]) + 1) if len(fft) > 1 else 0.0
        return pd.Series(out)

    fft_df = df.groupby('obs', sort=False).apply(fft_feats).reset_index()
    grouped = grouped.merge(fft_df, on='obs', how='left')

    # Zero-crossing rate
    def zero_crossings(block):
        out = {}
        for c in t_cols:
            x = block[c].values.astype(float) - block[c].mean()
            out[f'{c}_zcr'] = float(np.sum(np.diff(np.sign(x)) != 0))
        return pd.Series(out)

    zcr_df = df.groupby('obs', sort=False).apply(zero_crossings).reset_index()
    grouped = grouped.merge(zcr_df, on='obs', how='left')

    return grouped

In [4]:
train_agg = engineer_features(train_raw)
test_agg  = engineer_features(test_raw)

targets   = train_raw.groupby('obs')[['y_1', 'y_2']].first().reset_index()
train_agg = train_agg.merge(targets, on='obs')

print(f'Train: {train_agg.shape}')
print(f'Test:  {test_agg.shape}')

Train: (14420, 143)
Test:  (3450, 141)


## 3. Prepare Static Features + Target Encoding

In [5]:
drop_cols       = ['obs', 'sub_id_first', 'y_1', 'y_2']
cat_features_raw = [f'cat_{i}_first' for i in range(5)]

for c in cat_features_raw:
    le       = LabelEncoder()
    all_vals = pd.concat([train_agg[c], test_agg[c]]).astype(str)
    le.fit(all_vals)
    train_agg[c] = le.transform(train_agg[c].astype(str))
    test_agg[c]  = le.transform(test_agg[c].astype(str))

# Leakage-free GroupKFold target encoding
def target_encode_subject(train_df, test_df, target_col, group_col='sub_id_first', n_splits=5):
    train_df = train_df.copy()
    test_df  = test_df.copy()
    col_name = f'{group_col}_te_{target_col}'
    train_df[col_name] = np.nan
    gkf        = GroupKFold(n_splits=n_splits)
    sub_groups = train_df[group_col].values
    for tr_idx, va_idx in gkf.split(train_df, groups=sub_groups):
        means = train_df.iloc[tr_idx].groupby(group_col)[target_col].mean()
        train_df.iloc[va_idx, train_df.columns.get_loc(col_name)] = \
            train_df.iloc[va_idx][group_col].map(means)
    global_mean          = train_df[target_col].mean()
    train_df[col_name]   = train_df[col_name].fillna(global_mean)
    overall              = train_df.groupby(group_col)[target_col].mean()
    test_df[col_name]    = test_df[group_col].map(overall).fillna(global_mean)
    return train_df, test_df

for t in ['y_1', 'y_2']:
    train_agg, test_agg = target_encode_subject(train_agg, test_agg, t)

feature_cols = [c for c in train_agg.columns if c not in drop_cols]

X_all  = train_agg[feature_cols].replace([np.inf, -np.inf], np.nan).copy()
X_test = test_agg[feature_cols].replace([np.inf, -np.inf], np.nan).copy()
y_all  = train_agg[['y_1', 'y_2']].copy()
groups = train_agg['sub_id_first'].values

print(f'Feature count: {len(feature_cols)}')

Feature count: 141


## 4. Build Raw Sequences + Normalise

In [6]:
def build_sequences(raw_df, obs_order):
    """Returns array of shape (n_obs, 5, n_timesteps)."""
    t_cols     = [f't_{i}' for i in range(5)]
    raw_sorted = raw_df.sort_values(['obs', 'time'])
    seq_list   = []
    for obs in obs_order:
        block = raw_sorted[raw_sorted['obs'] == obs][t_cols].values  # (T, 5)
        seq_list.append(block.T)                                       # (5, T)
    return np.array(seq_list, dtype=np.float32)

train_obs_order = train_agg['obs'].values
test_obs_order  = test_agg['obs'].values

print('Building sequences...')
X_seq_train = build_sequences(train_raw, train_obs_order)
X_seq_test  = build_sequences(test_raw,  test_obs_order)

# Normalise sequences
seq_mean    = X_seq_train.mean(axis=(0, 2), keepdims=True)
seq_std     = X_seq_train.std(axis=(0, 2),  keepdims=True) + 1e-8
X_seq_train = (X_seq_train - seq_mean) / seq_std
X_seq_test  = (X_seq_test  - seq_mean) / seq_std

# Normalise static features
static_scaler   = StandardScaler()
X_static_train  = static_scaler.fit_transform(X_all.fillna(0).values).astype(np.float32)
X_static_test   = static_scaler.transform(X_test.fillna(0).values).astype(np.float32)

n_timesteps = X_seq_train.shape[2]
print(f'Sequence shape: {X_seq_train.shape}  →  (n_obs, n_signals, n_timesteps={n_timesteps})')
print(f'Static shape:   {X_static_train.shape}')

Building sequences...
Sequence shape: (14420, 5, 30)  →  (n_obs, n_signals, n_timesteps=30)
Static shape:   (14420, 141)


## 5. Dataset + Model

In [7]:
class TabSeqDataset(Dataset):
    def __init__(self, seq, static, targets=None):
        self.seq     = torch.tensor(seq)
        self.static  = torch.tensor(static)
        self.targets = torch.tensor(targets) if targets is not None else None

    def __len__(self): return len(self.seq)

    def __getitem__(self, i):
        if self.targets is not None:
            return self.seq[i], self.static[i], self.targets[i]
        return self.seq[i], self.static[i]


class TimeSeriesNet(nn.Module):
    def __init__(self, n_signals, n_static, n_out=2):
        super().__init__()

        self.cnn = nn.Sequential(
            nn.Conv1d(n_signals, 32,  kernel_size=3, padding=1),
            nn.BatchNorm1d(32),  nn.GELU(),
            nn.Conv1d(32, 64,   kernel_size=3, padding=1),
            nn.BatchNorm1d(64),  nn.GELU(),
            nn.Conv1d(64, 128,  kernel_size=3, padding=1),
            nn.BatchNorm1d(128), nn.GELU(),
            nn.AdaptiveAvgPool1d(4),   # → (B, 128, 4)
        )

        self.static_fc = nn.Sequential(
            nn.Linear(n_static, 128), nn.LayerNorm(128), nn.GELU(), nn.Dropout(0.3),
            nn.Linear(128, 64),       nn.GELU(),
        )

        self.head = nn.Sequential(
            nn.Linear(128 * 4 + 64, 256), nn.LayerNorm(256), nn.GELU(), nn.Dropout(0.3),
            nn.Linear(256, 64),           nn.GELU(),
            nn.Linear(64,  n_out),
        )

    def forward(self, x_seq, x_static):
        return self.head(torch.cat([self.cnn(x_seq).flatten(1), self.static_fc(x_static)], dim=1))


def mae_loss(pred, target):
    return torch.mean(torch.abs(pred - target))

## 6. Train with GroupKFold

In [8]:
N_FOLDS  = 5
NN_SEEDS = [42, 123, 2026]
TARGETS  = ['y_1', 'y_2']
EPOCHS   = 150
PATIENCE = 20
BATCH    = 64

gkf      = GroupKFold(n_splits=N_FOLDS)
n_train  = len(X_static_train)
n_test   = len(X_static_test)
n_static = X_static_train.shape[1]

# Accumulators — shape (n_obs,) per target
oof_y1   = np.zeros(n_train)
oof_y2   = np.zeros(n_train)
test_y1  = np.zeros(n_test)
test_y2  = np.zeros(n_test)

for seed in NN_SEEDS:
    print(f'\n=== SEED {seed} ===')
    for fold, (tr_idx, va_idx) in enumerate(gkf.split(X_static_train, groups=groups)):
        print(f'  Fold {fold+1}/{N_FOLDS}', end=' ')

        X_seq_tr   = X_seq_train[tr_idx]
        X_seq_va   = X_seq_train[va_idx]
        X_stat_tr  = X_static_train[tr_idx]
        X_stat_va  = X_static_train[va_idx]
        y_tr       = y_all.iloc[tr_idx].values.astype(np.float32)  # (n, 2)
        y_va       = y_all.iloc[va_idx].values.astype(np.float32)

        torch.manual_seed(seed)
        model     = TimeSeriesNet(n_signals=5, n_static=n_static).to(device)
        optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=1e-3)
        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

        train_dl  = DataLoader(TabSeqDataset(X_seq_tr, X_stat_tr, y_tr),
                               batch_size=BATCH, shuffle=True)

        best_mae, best_va_pred, best_te_pred = np.inf, None, None
        no_improve = 0

        for epoch in range(EPOCHS):
            # ── train ──
            model.train()
            for seq_b, stat_b, tgt_b in train_dl:
                seq_b, stat_b, tgt_b = seq_b.to(device), stat_b.to(device), tgt_b.to(device)
                optimizer.zero_grad()
                mae_loss(model(seq_b, stat_b), tgt_b).backward()
                nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()
            scheduler.step()

            # ── validate ──
            model.eval()
            with torch.no_grad():
                va_pred = model(
                    torch.tensor(X_seq_va).to(device),
                    torch.tensor(X_stat_va).to(device)
                ).cpu().numpy()   # (n_va, 2)

            va_mae = np.mean(np.abs(va_pred - y_va))

            if va_mae < best_mae:
                best_mae     = va_mae
                best_va_pred = va_pred.copy()
                with torch.no_grad():
                    best_te_pred = model(
                        torch.tensor(X_seq_test).to(device),
                        torch.tensor(X_static_test).to(device)
                    ).cpu().numpy()
                no_improve = 0
            else:
                no_improve += 1
                if no_improve >= PATIENCE:
                    break

        print(f'| best val MAE: {best_mae:.4f}  (stopped epoch {epoch+1})')

        oof_y1[va_idx] += best_va_pred[:, 0]
        oof_y2[va_idx] += best_va_pred[:, 1]
        test_y1        += best_te_pred[:, 0] / (N_FOLDS * len(NN_SEEDS))
        test_y2        += best_te_pred[:, 1] / (N_FOLDS * len(NN_SEEDS))

# Average OOF across seeds
oof_y1 /= len(NN_SEEDS)
oof_y2 /= len(NN_SEEDS)

print(f'\nFinal OOF MAE — y_1: {mean_absolute_error(y_all["y_1"], oof_y1):.4f}')
print(f'Final OOF MAE — y_2: {mean_absolute_error(y_all["y_2"], oof_y2):.4f}')
print(f'Average:             {(mean_absolute_error(y_all["y_1"], oof_y1) + mean_absolute_error(y_all["y_2"], oof_y2)) / 2:.4f}')


=== SEED 42 ===
  Fold 1/5 | best val MAE: 4.1584  (stopped epoch 68)
  Fold 2/5 | best val MAE: 4.4554  (stopped epoch 73)
  Fold 3/5 | best val MAE: 4.1058  (stopped epoch 63)
  Fold 4/5 | best val MAE: 4.2065  (stopped epoch 48)
  Fold 5/5 | best val MAE: 4.0726  (stopped epoch 58)

=== SEED 123 ===
  Fold 1/5 | best val MAE: 4.1429  (stopped epoch 65)
  Fold 2/5 | best val MAE: 4.4728  (stopped epoch 46)
  Fold 3/5 | best val MAE: 4.1135  (stopped epoch 83)
  Fold 4/5 | best val MAE: 4.2064  (stopped epoch 64)
  Fold 5/5 | best val MAE: 4.0576  (stopped epoch 93)

=== SEED 2026 ===
  Fold 1/5 | best val MAE: 4.1375  (stopped epoch 70)
  Fold 2/5 | best val MAE: 4.4849  (stopped epoch 82)
  Fold 3/5 | best val MAE: 4.1397  (stopped epoch 55)
  Fold 4/5 | best val MAE: 4.2151  (stopped epoch 69)
  Fold 5/5 | best val MAE: 4.0884  (stopped epoch 78)

Final OOF MAE — y_1: 4.8519
Final OOF MAE — y_2: 3.4115
Average:             4.1317


## 7. Save Outputs

In [9]:
# Save OOF + test preds as .npy so you can load them in your main notebook
np.save('nn_oof_y1.npy',  oof_y1)
np.save('nn_oof_y2.npy',  oof_y2)
np.save('nn_test_y1.npy', test_y1)
np.save('nn_test_y2.npy', test_y2)

# Optional: standalone submission
submission = pd.DataFrame({'obs': test_agg['obs'], 'y_1': test_y1, 'y_2': test_y2})
submission.to_csv('nn_submission.csv', index=False)

print('Saved:')
print('  nn_oof_y1.npy, nn_oof_y2.npy   ← load these into your main notebook for stacking')
print('  nn_test_y1.npy, nn_test_y2.npy')
print('  nn_submission.csv')

Saved:
  nn_oof_y1.npy, nn_oof_y2.npy   ← load these into your main notebook for stacking
  nn_test_y1.npy, nn_test_y2.npy
  nn_submission.csv
